# 23N Field Tests #12 · Indian-language dictation: a paid API against free open models, on public test clips

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23nlabs/field-tests/blob/ft12-dictation-indian-languages/ft12-dictation-indian-languages/ft12_dictation_indian_languages.ipynb)

**Question.** Apps that let people type by voice in Gujarati, Hindi or Tamil can run a free open speech model on
their own GPU or pay a dictation API per minute. On the *same public test clips*, how often does each get the words
wrong, and what does it cost?

**Why now.** On 23 Nov 2026 Sarvam AI released *Sarvam Dictate*, a dictation API for English and 22 Indian languages
priced at ₹0.50 per minute of audio. Its announcement reports fewer character edits than other commercial systems on
real dictations from its own app, a test set that is not public. This notebook runs the API on a public test set,
next to free open models, so anyone can repeat the comparison.

**Method.** Up to 100 clips per language from Google's FLEURS test set (read speech; the same sentences in every
language), in Gujarati, Hindi, Tamil and English. Each clip goes to Sarvam Dictate once (one call returns both the
raw speech-recognition `transcript` and the edited `text`) and to two open models on a free Colab GPU. Every output
goes through one script-safe normaliser and is scored with word error rate (WER) and character error rate (CER).

| System | What it is | Licence / price |
|---|---|---|
| Sarvam Dictate (`dictate-v1`), raw `transcript` | commercial API (Saaras v4 recognition) | ₹0.50 per audio minute; ₹100 free credit for verified sign-ups (per Sarvam's pricing page) |
| Sarvam Dictate, edited `text` (reported separately) | the same call, after its language-model editing step | same call, no extra cost |
| `openai/whisper-large-v3-turbo` | open model, ~0.81B | MIT |
| `facebook/mms-1b-all` (language adapters) | open model, ~1B | **CC-BY-NC 4.0 (non-commercial)** |

**Data.** [FLEURS](https://huggingface.co/datasets/google/fleurs) (Google), test split, configs `gu_in`, `hi_in`,
`ta_in`, `en_us`. Licence **CC BY 4.0**, no login. Audio is downloaded at run time and not redistributed.

**You need a Sarvam API key** (free sign-up at dashboard.sarvam.ai; the free credit covers this run, about
₹40–60 of audio). Put it in Colab's 🔑 Secrets panel as `SARVAM_API_KEY` (or set the environment variable, or a
Kaggle secret of the same name). Without a key the open models still run, and the notebook says loudly that the
comparison is incomplete and must not be posted.

**Runtime.** Free Colab T4: about 15–20 minutes for Run all.

*Independent study on public data by 23N AI Labs. Not affiliated with or endorsed by Sarvam AI, Google, OpenAI,
Meta or any model author mentioned here. The API was used as any paying developer would use it.*

## 1 · Setup

In [ ]:
# CELL: system_deps
# libraqm gives Pillow complex-script shaping, which Gujarati, Hindi and Tamil need (matras and conjuncts).
!apt-get -qq install -y libraqm0 libfribidi0 > /dev/null 2>&1 || echo "apt not available, continuing"
%pip install -q jiwer soundfile gradio requests

In [ ]:
# CELL: config
import os, sys, io, re, json, math, time, textwrap, unicodedata, subprocess, warnings, tarfile, gc, collections, zlib
import threading, concurrent.futures as cf
import urllib.request
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", message="Glyph .* missing")

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

# ---- Test switch. OFF by default. Only for build checks without internet, GPU or API key: it replaces the FLEURS
# download with a few synthetic clips, and the models and the API with deterministic stand-ins. Output is NOT real.
SMOKE_TEST = os.environ.get("FT12_SMOKE_TEST", "0") == "1"

# ---- Assumptions and knobs (all named, all printed with the result) ----
LANGS = {  # FLEURS config -> (label, Whisper code, MMS adapter, main script, Sarvam language_code)
    "gu_in": ("Gujarati", "gu", "guj", "Gujarati", "gu-IN"),
    "hi_in": ("Hindi", "hi", "hin", "Devanagari", "hi-IN"),
    "ta_in": ("Tamil", "ta", "tam", "Tamil", "ta-IN"),
    "en_us": ("English", "en", "eng", "Latin", "en-IN"),
}
OPEN_MODELS = [  # key, Hugging Face id, family, licence
    ("whisper-large-v3-turbo", "openai/whisper-large-v3-turbo", "whisper", "MIT"),
    ("mms-1b-all", "facebook/mms-1b-all", "mms", "CC-BY-NC-4.0 (non-commercial)"),
]
API_RAW, API_EDIT = "sarvam-dictate (raw)", "sarvam-dictate (edited)"

DICTATE_URL = "https://api.sarvam.ai/v2/dictate"   # clip endpoint, max 30 s per request (Sarvam docs)
DICTATE_MODEL = "dictate-v1"      # documented alias; the served snapshot is echoed back and printed
DICTATE_MODE = "transcribe"       # default mode
DICTATE_PRESET = "clean"          # style preset for the edited text; the raw transcript is unaffected
PRICE_INR_PER_MIN = 0.50          # pay-as-you-go list price on 23 Nov 2026, excl. GST (assumption: check before quoting)
DICTATE_RPM = 50                  # stay under the 60 requests/min pay-as-you-go limit
DICTATE_WORKERS = 3
MAX_API_MINUTES = 150             # hard budget per run (≈ ₹75 at list price); the run stops calling the API past it

N_PER_LANG = 100          # clips per language (same sentence IDs across languages where possible)
N_MIN = 80                # fewer usable clips than this per language → the run is flagged
SEED = 23
MAX_AUDIO_S = 29.5        # the API takes ≤30 s per clip; longer clips are excluded for every system (fair comparison)
BATCH = 16
MAX_NEW_TOKENS = 225
EN_WER_CEILING = 0.15     # assumption: on read English a large modern recogniser should be far below this
API_VS_OPEN_LIMIT = 2.0   # API raw WER above this × the best open model's in a language → implausible, check the call
MAX_FAIL_SHARE = 0.05     # API calls that failed, came back degraded or in the wrong language: above this → flagged
N_BOOT = 1000

# ---- Where things live (this branch until it is merged) ----
REPO_REF = "ft12-dictation-indian-languages"
RAW = f"https://raw.githubusercontent.com/23nlabs/field-tests/{REPO_REF}"
FLEURS_REPO = "google/fleurs"
FT_NUM = "12"
DATA_CREDIT = "Data: Google FLEURS test set, CC BY 4.0"

# ---- API key: env var, Colab secret or Kaggle secret. Never printed, never saved. ----
SARVAM_KEY = os.environ.get("SARVAM_API_KEY", "")
if not SARVAM_KEY and IN_COLAB:
    try:
        from google.colab import userdata
        SARVAM_KEY = userdata.get("SARVAM_API_KEY") or ""
    except Exception as e:
        print("Colab secret SARVAM_API_KEY not available:", type(e).__name__)
if not SARVAM_KEY and IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        SARVAM_KEY = UserSecretsClient().get_secret("SARVAM_API_KEY") or ""
    except Exception as e:
        print("Kaggle secret SARVAM_API_KEY not available:", type(e).__name__)
API_ON = bool(SARVAM_KEY) or (SMOKE_TEST and os.environ.get("FT12_SMOKE_API", "1") == "1")

# ---- Brand kit ----
if not os.path.exists("brand23n.py"):
    urllib.request.urlretrieve(f"{RAW}/brand/brand23n.py", "brand23n.py")
import brand23n as B
B.setup()
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    HARDWARE = torch.cuda.get_device_name(0) if HAS_GPU else "CPU"
except Exception:
    HAS_GPU, HARDWARE = False, "CPU"
if SMOKE_TEST:
    HARDWARE = "CPU (stand-in)"
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE, "| GPU:", HAS_GPU, "| API key found:", bool(SARVAM_KEY),
      "| SMOKE_TEST:", SMOKE_TEST)
if not API_ON:
    print("!! No SARVAM_API_KEY: the API arm will be skipped. Open models still run, but the comparison is incomplete.")
if not HAS_GPU and not SMOKE_TEST:
    print("!! No GPU: the open models will be very slow. Switch to a T4 (Runtime → Change runtime type).")
if SMOKE_TEST:
    print("!! SMOKE TEST MODE: synthetic clips, stand-in models and a stand-in API. These results are NOT real. Do NOT post.")

In [ ]:
# CELL: drive_mount
# Every run's outputs are copied to your Google Drive at the end (see "Save to Drive"). Mounting happens here, at the
# start, so the one-time Google sign-in pop-up appears while you're still at the screen, not 20 minutes in.
SAVE_TO_DRIVE = True                                         # set False to keep outputs only in this Colab session
DRIVE_ROOT = "/content/drive/MyDrive/23N Field Tests"       # outputs go to DRIVE_ROOT/FT12/<date_time>[_label]
SAVE_DIR_OVERRIDE = os.environ.get("FT12_SAVE_DIR", "")     # for testing outside Colab: copy to this local folder
DRIVE_OK = False
if SAVE_DIR_OVERRIDE:
    DRIVE_OK = True
elif SAVE_TO_DRIVE and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OK = True
    except Exception as e:
        print("!! Google Drive not mounted, outputs will stay in this session:", type(e).__name__, str(e)[:150])
elif SAVE_TO_DRIVE:
    print("Not on Colab: outputs stay in the working directory (set FT12_SAVE_DIR to copy them elsewhere).")
print("Save outputs to Drive:", DRIVE_OK)

In [ ]:
# CELL: fonts
# IBM Plex has no Indic glyphs, so Gujarati, Hindi and Tamil strings are drawn with Noto Sans (SIL OFL, from the
# Google Fonts GitHub repo; each also covers Latin and digits) through Pillow + libraqm, then pasted into the figure.
from PIL import Image, ImageDraw, ImageFont, features
os.makedirs("fonts", exist_ok=True)
FONT_URLS = {
    "Gujarati": ("fonts/NotoSansGujarati-VF.ttf",
                 "https://raw.githubusercontent.com/google/fonts/main/ofl/notosansgujarati/NotoSansGujarati%5Bwdth%2Cwght%5D.ttf"),
    "Devanagari": ("fonts/NotoSansDevanagari-VF.ttf",
                   "https://raw.githubusercontent.com/google/fonts/main/ofl/notosansdevanagari/NotoSansDevanagari%5Bwdth%2Cwght%5D.ttf"),
    "Tamil": ("fonts/NotoSansTamil-VF.ttf",
              "https://raw.githubusercontent.com/google/fonts/main/ofl/notosanstamil/NotoSansTamil%5Bwdth%2Cwght%5D.ttf"),
}
for path, url in FONT_URLS.values():
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
HAS_RAQM = features.check("raqm")
if not HAS_RAQM:
    print("WARNING: Pillow has no libraqm, so Indic text will be drawn WITHOUT shaping.\n"
          "Fix: run `!apt-get install -y libraqm0 libfribidi0`, then Runtime → Restart session, then Run all.")

SCRIPT_RANGES = {"Gujarati": ("઀", "૿"), "Devanagari": ("ऀ", "ॿ"), "Tamil": ("஀", "௿")}
def script_share(s, script):
    """Share of letters in `s` that belong to `script` (Latin = ASCII letters)."""
    letters = [c for c in str(s) if c.isalpha() or unicodedata.category(c) in ("Mn", "Mc")]
    if not letters:
        return 0.0
    if script == "Latin":
        return sum(c.isascii() for c in letters) / len(letters)
    lo, hi = SCRIPT_RANGES[script]
    return sum(lo <= c <= hi for c in letters) / len(letters)

_font_cache = {}
def _pil_font(px, bold=False, script="Gujarati"):
    key = (int(px), bold, script)
    if key not in _font_cache:
        layout = ImageFont.Layout.RAQM if HAS_RAQM else ImageFont.Layout.BASIC
        f = ImageFont.truetype(FONT_URLS[script][0], int(px), layout_engine=layout)
        try:
            f.set_variation_by_axes([600 if bold else 400, 100])   # axes: wght, wdth
        except Exception:
            pass
        _font_cache[key] = f
    return _font_cache[key]

def text_image(text, px, color=B.INK, bold=False, max_width=None, line_gap=1.3, max_lines=None):
    """Render text in any of our scripts to an RGBA array with proper shaping, wrapped to max_width pixels."""
    script = max(SCRIPT_RANGES, key=lambda s: script_share(text, s))
    font = _pil_font(px, bold, script)
    lines = []
    for para in str(text).split("\n"):
        cur = ""
        for w in para.split(" "):
            trial = (cur + " " + w).strip()
            if max_width and cur and font.getlength(trial) > max_width:
                lines.append(cur); cur = w
            else:
                cur = trial
        lines.append(cur)
    if max_lines and len(lines) > max_lines:
        lines = lines[:max_lines]; lines[-1] = lines[-1].rstrip(" .,") + " …"
    lh = int(px * line_gap * 1.15)
    width = int(max(font.getlength(l) for l in lines)) + 8
    img = Image.new("RGBA", (max(width, 1), lh * len(lines) + int(px * 0.45)), (0, 0, 0, 0))
    d = ImageDraw.Draw(img)
    for i, l in enumerate(lines):
        d.text((2, i * lh), l, font=font, fill=color)
    return np.asarray(img)

def put_text_img(fig, text, x, y, px, color=B.INK, bold=False, ha="left", max_width=None, alpha=1.0, max_lines=None):
    """Place shaped text on a figure. x, y = figure fraction of the top-left anchor. Returns height in figure fraction."""
    arr = text_image(text, px, color, bold, max_width, max_lines=max_lines)
    if alpha < 1:
        arr = arr.copy(); arr[..., 3] = (arr[..., 3] * alpha).astype(np.uint8)
    W, H = fig.get_size_inches() * fig.dpi
    h, w = arr.shape[:2]
    xo = x * W - (w / 2 if ha == "center" else (w if ha == "right" else 0))
    fig.figimage(arr, xo=int(xo), yo=int(y * H - h), origin="upper", zorder=10)
    return h / H

_chk = plt.figure(figsize=(9, 2.2), dpi=100)
put_text_img(_chk, "ગુજરાતી તપાસ: સ્પષ્ટ ઉચ્ચારણ, ક્ષ ત્ર જ્ઞ ૧૨૩", 0.02, 0.95, 30, B.INK, bold=True)
put_text_img(_chk, "हिंदी जाँच: क्षत्रिय ज्ञान श्री", 0.02, 0.63, 30, B.INK, bold=True)
put_text_img(_chk, "தமிழ் சோதனை: க்ஷ ஸ்ரீ கொ", 0.02, 0.31, 30, B.INK, bold=True)
_chk.savefig("script_render_check.png", facecolor=B.PAPER); plt.close(_chk)
print("libraqm shaping:", HAS_RAQM, "→ saved script_render_check.png (open it to eyeball the conjuncts)")

## 2 · Data: the same sentences, read aloud in four languages (FLEURS test set)

Each FLEURS sentence is a translation of the same FLoRes-101 sentence, so a sentence ID means the same content in
every language. The notebook downloads each language's test transcript (`test.tsv`) and test audio
(`audio/test.tar.gz`, 16 kHz WAV) from the Hugging Face dataset repo, keeps one recording per sentence, drops
recordings longer than `MAX_AUDIO_S` (the API takes at most 30 s per clip, so every system gets the same clips), and
picks up to `N_PER_LANG` sentence IDs present in all four languages (seeded, so the selection is reproducible).

In [ ]:
# CELL: data
def _hf_path(lang, kind):
    """Locate FLEURS files for `lang` in the HF repo. kind = 'tsv' or 'audio'. Tries the documented layout first."""
    from huggingface_hub import hf_hub_download, HfApi
    first = f"data/{lang}/test.tsv" if kind == "tsv" else f"data/{lang}/audio/test.tar.gz"
    try:
        return hf_hub_download(FLEURS_REPO, first, repo_type="dataset")
    except Exception as e:
        print(f"  {first} not found directly ({type(e).__name__}); searching the repo file list")
    files = HfApi().list_repo_files(FLEURS_REPO, repo_type="dataset")
    want = (lambda f: f.endswith("test.tsv")) if kind == "tsv" else (lambda f: f.endswith("test.tar.gz"))
    cands = [f for f in files if f"/{lang}/" in f and want(f) and "parquet" not in f]
    if not cands:
        raise FileNotFoundError(f"no {kind} file for {lang} in {FLEURS_REPO}")
    return hf_hub_download(FLEURS_REPO, cands[0], repo_type="dataset")

def read_tsv(path):
    """FLEURS TSV, no header: id, file name, raw transcription, normalised transcription, chars, num_samples, gender."""
    rows = []
    for line in open(path, encoding="utf-8"):
        p = line.rstrip("\n").split("\t")
        if len(p) < 7 or not p[5].strip().isdigit():
            continue
        rows.append({"sid": p[0].strip(), "file": p[1].strip(), "raw": p[2].strip(), "norm_ref": p[3].strip(),
                     "num_samples": int(p[5]), "gender": p[6].strip().upper()})
    return pd.DataFrame(rows)

def load_wavs(tar_path, wanted):
    """Extract only the wanted file names from a FLEURS audio archive. Returns {file name: (float32 mono, sr)}."""
    import soundfile as sf
    out = {}
    with tarfile.open(tar_path, "r:gz") as tar:
        for m in tar:
            name = os.path.basename(m.name)
            if m.isfile() and name in wanted:
                a, sr = sf.read(io.BytesIO(tar.extractfile(m).read()), dtype="float32")
                out[name] = (a.mean(axis=1) if a.ndim > 1 else a, sr)
                if len(out) == len(wanted):
                    break
    return out

def to_16k(a, sr):
    if sr == 16000:
        return a
    from scipy.signal import resample_poly
    g = math.gcd(int(sr), 16000)
    return resample_poly(a, 16000 // g, int(sr) // g).astype(np.float32)

def load_fleurs():
    rng = np.random.default_rng(SEED)
    tables, n_long = {}, 0
    for lang in LANGS:
        t = read_tsv(_hf_path(lang, "tsv"))
        long_ = t.num_samples / 16000 > MAX_AUDIO_S     # FLEURS audio is 16 kHz; re-checked after decoding
        n_long += int(long_.sum())
        t = t[~long_].sample(frac=1.0, random_state=SEED).drop_duplicates("sid")   # one recording per sentence
        tables[lang] = t.set_index("sid")
        print(f"{lang}: {len(t)} test sentences with a recording ≤ {MAX_AUDIO_S:g}s")
    common = sorted(set.intersection(*[set(t.index) for t in tables.values()]))
    parallel = len(common) >= N_PER_LANG
    common_ids = list(rng.choice(common, N_PER_LANG, replace=False)) if parallel else None
    if not parallel:
        print(f"!! Only {len(common)} sentence IDs are shared by all four languages: sampling each language separately")
    clips = []
    for lang, t in tables.items():
        ids = common_ids if parallel else list(rng.choice(t.index.values, min(N_PER_LANG, len(t)), replace=False))
        sel = t.loc[ids]
        wavs = load_wavs(_hf_path(lang, "audio"), set(sel.file))
        for sid, r in sel.iterrows():
            if r.file not in wavs:
                continue
            a, sr = wavs[r.file]
            clips.append({"lang": lang, "sid": sid, "file": r.file, "ref_raw": r.raw, "gender": r.gender,
                          "sr_in": sr, "audio": to_16k(a, sr)})
    print(f"recordings over {MAX_AUDIO_S:g}s excluded (all languages): {n_long}")
    return pd.DataFrame(clips), parallel

def load_stand_in():
    """SMOKE_TEST only: no network. Synthetic noise bursts with short sentences we wrote ourselves (not FLEURS)."""
    sents = {
        "gu_in": ["આજે અમદાવાદમાં વરસાદ પડ્યો", "બજારમાં શાકભાજીના ભાવ વધ્યા છે", "ટ્રેન સવારે ૧૦ વાગ્યે આવશે",
                  "શાળામાં ૨૫ બાળકો છે", "ખેડૂતો પાણીની રાહ જુએ છે", "આ રસ્તો ગાંધીનગર તરફ જાય છે"],
        "hi_in": ["आज अहमदाबाद में बारिश हुई", "बाज़ार में सब्ज़ियों के दाम बढ़े हैं", "ट्रेन सुबह १० बजे आएगी",
                  "स्कूल में २५ बच्चे हैं", "किसान पानी का इंतज़ार कर रहे हैं", "यह सड़क गांधीनगर की ओर जाती है"],
        "ta_in": ["இன்று அகமதாபாத்தில் மழை பெய்தது", "சந்தையில் காய்கறி விலை உயர்ந்துள்ளது", "ரயில் காலை 10 மணிக்கு வரும்",
                  "பள்ளியில் 25 குழந்தைகள் உள்ளனர்", "விவசாயிகள் தண்ணீருக்காக காத்திருக்கிறார்கள்", "இந்த சாலை காந்திநகர் நோக்கி செல்கிறது"],
        "en_us": ["It rained in Ahmedabad today", "Vegetable prices have gone up in the market",
                  "The train will arrive at 10 in the morning", "There are 25 children in the school",
                  "Farmers are waiting for water", "This road goes towards Gandhinagar"],
    }
    rng = np.random.default_rng(SEED); clips = []
    for lang, ss in sents.items():
        for i in range(12):
            s = ss[i % len(ss)]; n = int(16000 * (1.2 + 0.25 * len(s.split())))
            clips.append({"lang": lang, "sid": f"S{i:02d}", "file": f"synthetic_{lang}_{i:02d}.wav", "ref_raw": s,
                          "gender": "MALE" if i % 2 else "FEMALE", "sr_in": 16000,
                          "audio": (0.05 * rng.standard_normal(n)).astype(np.float32)})
    return pd.DataFrame(clips), True

t0 = time.time()
CLIPS, PARALLEL = load_stand_in() if SMOKE_TEST else load_fleurs()
CLIPS["dur_s"] = CLIPS.audio.map(len) / 16000
N_LONG_DECODED = int((CLIPS.dur_s > MAX_AUDIO_S).sum())        # should be 0; checked in the sanity checks
CLIPS = CLIPS[CLIPS.dur_s <= MAX_AUDIO_S].reset_index(drop=True)
N_CLIPS = CLIPS.groupby("lang").size().reindex(list(LANGS)).fillna(0).astype(int)
DATA_TABLE = CLIPS.groupby("lang").agg(clips=("sid", "size"), minutes=("dur_s", lambda s: s.sum() / 60),
                                       median_s=("dur_s", "median"),
                                       female=("gender", lambda g: (g == "FEMALE").mean())).reindex(list(LANGS))
API_MINUTES_PLANNED = float(CLIPS.dur_s.sum() / 60)
print(DATA_TABLE.round(2).to_string())
print(f"\nParallel sentences across languages: {PARALLEL} · {len(CLIPS)} clips · {API_MINUTES_PLANNED:.0f} min of audio "
      f"(≈ ₹{API_MINUTES_PLANNED * PRICE_INR_PER_MIN:.0f} of API time at list price) · loaded in {time.time() - t0:.0f}s")

fig, ax = plt.subplots(figsize=(9, 3.8))
for i, lang in enumerate(LANGS):
    ax.hist(CLIPS[CLIPS.lang == lang].dur_s, bins=20, alpha=0.75, color=B.SERIES_COLORS[i], label=LANGS[lang][0])
ax.set_xlabel("clip length (seconds)"); ax.set_ylabel("clips"); ax.legend()
B.chart_title(ax, "The test clips", f"{int(N_CLIPS.min())}–{int(N_CLIPS.max())} clips per language · FLEURS test split")
B.source_note(fig, "23N Field Test #12 · " + DATA_CREDIT + " · Independent study on public data.")
plt.tight_layout(); plt.show()

## 3 · The systems

**Open models** are loaded one at a time on the Colab GPU, run on every language, timed and freed. Whisper gets the
language forced (`language="gu"`, `task="transcribe"`); MMS swaps in its Gujarati, Hindi, Tamil or English adapter.
Decoding is greedy.

**The API** gets each clip once as a 16 kHz, 16-bit WAV with the language stated (`language_code="gu-IN"` and so on,
as Sarvam's reference advises), `mode="transcribe"` and `style={"preset": "clean"}`. One call returns the raw
`transcript` ("the words that speech recognition heard") and the edited `text`. The notebook:
- caches every response in `ft12_api_cache.json`, so re-running never bills the same clip twice;
- stays under the rate limit (`DICTATE_RPM`), retries on 429/5xx with back-off, and stops at `MAX_API_MINUTES`;
- records latency, the served model snapshot, the language the API reports, and the `degraded` flag.

The key is read from secrets and never printed or saved.

In [ ]:
# CELL: models
def _free():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

def load_asr(key, hf_id, family):
    """Returns (transcribe(list of 16 kHz arrays, fleurs lang) -> list of str, params in billions)."""
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if dev == "cuda" else torch.float32
    if family == "whisper":
        from transformers import WhisperForConditionalGeneration, WhisperProcessor
        proc = WhisperProcessor.from_pretrained(hf_id)
        model = WhisperForConditionalGeneration.from_pretrained(hf_id, torch_dtype=dtype).to(dev).eval()
        model.generation_config.forced_decoder_ids = None    # the language/task come from generate() below
        model.config.forced_decoder_ids = None
        def transcribe(arrs, lang):
            feats = proc.feature_extractor(arrs, sampling_rate=16000, return_tensors="pt").input_features
            with torch.no_grad():
                ids = model.generate(feats.to(dev, dtype), language=LANGS[lang][1], task="transcribe",
                                     max_new_tokens=MAX_NEW_TOKENS, num_beams=1, do_sample=False)
            return [s.strip() for s in proc.batch_decode(ids, skip_special_tokens=True)]
    else:
        from transformers import AutoProcessor, Wav2Vec2ForCTC
        proc = AutoProcessor.from_pretrained(hf_id)
        model = Wav2Vec2ForCTC.from_pretrained(hf_id).to(dev).eval()
        state = {"lang": None}
        def transcribe(arrs, lang):
            code = LANGS[lang][2]
            if state["lang"] != code:
                proc.tokenizer.set_target_lang(code); model.load_adapter(code); state["lang"] = code
            inp = proc(arrs, sampling_rate=16000, return_tensors="pt", padding=True)
            mask = inp.get("attention_mask", None)
            with torch.no_grad():
                logits = model(inp.input_values.to(dev), attention_mask=None if mask is None else mask.to(dev)).logits
            return [s.strip() for s in proc.batch_decode(logits.argmax(-1))]
    params = sum(p.numel() for p in model.parameters()) / 1e9
    return transcribe, params

def _corrupt(ref, key, rate, lang):
    """SMOKE_TEST helper: the reference with seeded word drops and reversals at a made-up rate."""
    r = np.random.default_rng(zlib.crc32(f"{key}|{lang}|{ref}".encode()))
    words = [w for w in ref.split() if r.random() > rate * 0.5]
    return " ".join(w[::-1] if r.random() < rate * 0.5 else w for w in words)

LANG_MULT = {"gu_in": 1.0, "hi_in": 0.6, "ta_in": 0.9, "en_us": 0.25}
def load_stand_in_asr(key, hf_id, family):
    """SMOKE_TEST only: says nothing about any real model."""
    rate = {"whisper-large-v3-turbo": 0.35, "mms-1b-all": 0.3}[key]
    def transcribe(arrs, lang):
        return [_corrupt(CLIPS_BY_ID.get(id(a), ""), key, rate * LANG_MULT[lang], lang) for a in arrs]
    return transcribe, 0.0

if SMOKE_TEST:
    CLIPS_BY_ID = {id(a): s for a, s in zip(CLIPS.audio, CLIPS.ref_raw)}
print("Open models:", [m[0] for m in OPEN_MODELS], "| API arm:", "on" if API_ON else "OFF (no key)")

In [ ]:
# CELL: api_client
import requests, soundfile as sf
CACHE_PATH = "ft12_api_cache.json"
try:
    API_CACHE = json.load(open(CACHE_PATH, encoding="utf-8"))
except Exception:
    API_CACHE = {}

class _RateLimiter:
    def __init__(self, per_min):
        self.gap, self.lock, self.next = 60.0 / per_min, threading.Lock(), 0.0
    def wait(self):
        with self.lock:
            now = time.time(); t = max(now, self.next); self.next = t + self.gap
        time.sleep(max(0.0, t - now))
_LIMIT = _RateLimiter(DICTATE_RPM)

def _wav_bytes(a):
    buf = io.BytesIO(); sf.write(buf, a, 16000, format="WAV", subtype="PCM_16"); return buf.getvalue()

def dictate_call(audio, lang, tries=5):
    """One clip → dict(transcript, text, language_code, degraded, model, audio_s, latency_s) or dict(error=...)."""
    data = {"model": DICTATE_MODEL, "language_code": LANGS[lang][4], "mode": DICTATE_MODE,
            "style": json.dumps({"preset": DICTATE_PRESET})}
    wav = _wav_bytes(audio); err = None
    for k in range(tries):
        _LIMIT.wait(); t = time.time()
        try:
            r = requests.post(DICTATE_URL, headers={"api-subscription-key": SARVAM_KEY},
                              files={"file": ("clip.wav", wav, "audio/wav")}, data=data, timeout=90)
        except requests.RequestException as e:
            err = f"{type(e).__name__}"; time.sleep(2 ** k); continue
        lat = time.time() - t
        if r.status_code == 429 or r.status_code >= 500:
            err = f"HTTP {r.status_code}"
            time.sleep(float(r.headers.get("Retry-After", 2 ** k))); continue
        if r.status_code != 200:
            return {"error": f"HTTP {r.status_code}: {r.text[:300]}"}    # 4xx: our request is wrong; don't retry
        j = r.json(); u = j.get("usage") or {}
        return {"transcript": j.get("transcript") or "", "text": j.get("text") or "",
                "language_code": j.get("language_code"), "degraded": j.get("degraded"), "model": j.get("model"),
                "audio_s": u.get("audio_duration_s"), "latency_s": lat}
    return {"error": err or "failed"}

def stand_in_dictate(ref, lang):
    """SMOKE_TEST only: a made-up 'raw' with errors and an 'edited' version with punctuation. Not the real API."""
    raw = _corrupt(ref, "api", 0.2 * LANG_MULT[lang], lang)
    w = raw.split()
    if len(w) > 4 and zlib.crc32(ref.encode()) % 3 == 0:
        w = w[:-1]                                        # mimic an edit that drops a word
    text = (" ".join(w)[:1].upper() + " ".join(w)[1:] + ".") if w else raw
    return {"transcript": raw, "text": text, "language_code": LANGS[lang][4], "degraded": None,
            "model": "stand-in", "audio_s": None, "latency_s": 0.8 + 0.05 * len(ref.split())}

def run_api(clips):
    """Send every clip once (cached), within the minute budget. Returns {row index: response}."""
    out, todo, billed = {}, [], 0.0
    for i, r in clips.iterrows():
        key = f"{DICTATE_MODEL}|{DICTATE_MODE}|{DICTATE_PRESET}|{r.lang}|{r.file}"
        if key in API_CACHE and "error" not in API_CACHE[key]:
            out[i] = API_CACHE[key]
        else:
            todo.append((i, key, r))
    print(f"API: {len(out)} cached, {len(todo)} to send")
    def job(item):
        i, key, r = item
        res_ = stand_in_dictate(r.ref_raw, r.lang) if SMOKE_TEST else dictate_call(r.audio, r.lang)
        return i, key, res_, r.dur_s
    with cf.ThreadPoolExecutor(DICTATE_WORKERS) as ex:
        futs = []
        for item in todo:
            if billed + item[2].dur_s / 60 > MAX_API_MINUTES:
                print(f"!! Budget of {MAX_API_MINUTES} API minutes reached: {len(todo) - len(futs)} clips not sent"); break
            billed += item[2].dur_s / 60; futs.append(ex.submit(job, item))
        for n, f in enumerate(cf.as_completed(futs), 1):
            i, key, res_, dur = f.result(); out[i] = res_; API_CACHE[key] = res_
            if n % 50 == 0:
                print(f"  {n}/{len(futs)} sent")
    if not SMOKE_TEST:
        json.dump(API_CACHE, open(CACHE_PATH, "w", encoding="utf-8"), ensure_ascii=False)
    return out, billed

print("API client ready:", DICTATE_URL, "| model", DICTATE_MODEL, "| mode", DICTATE_MODE, "| preset", DICTATE_PRESET)

## 4 · Method: one normaliser for every system, then WER and CER

The FLEURS reference and every output go through the same normaliser: Unicode NFC, lower case, Gujarati, Devanagari
and Tamil digits mapped to 0–9, punctuation and symbols turned into spaces, whitespace collapsed. Vowel signs and
viramas are **kept** (Whisper's `BasicTextNormalizer` deletes them, which splits Indic words apart).

- **WER** = (substitutions + deletions + insertions) / reference words, pooled over all clips of a language.
- **CER** = the same on characters; kinder to long, inflected words. Both are printed.
- **Headline uses the API's raw `transcript`.** The edited `text` is built for people reading a message, not for
  matching a verbatim reference: it can drop fillers, fix grammar or write numbers as digits, and WER counts each of
  those as an error even when a reader would prefer it. It is scored and reported separately, with how much the
  editor changed.
- **Confidence intervals**: 95% bootstrap over clips; the API-versus-open gap uses a *paired* bootstrap (the same
  clips resampled for both).

In [ ]:
# CELL: scoring
import jiwer
DIGITS = str.maketrans("૦૧૨૩૪૫૬૭૮૯०१२३४५६७८९௦௧௨௩௪௫௬௭௮௯", "012345678901234567890123456789")

def normalise(s):
    s = unicodedata.normalize("NFC", str(s)).lower().translate(DIGITS)
    s = "".join(" " if unicodedata.category(c)[0] in "PSZC" and c != " " else c for c in s)
    return re.sub(r"\s+", " ", s).strip()

def utt_errors(ref, hyp):
    """Word and character error counts for one clip, plus the word alignment."""
    if not hyp:     # an empty output = every reference word (and character) deleted
        return dict(w_err=len(ref.split()), w_ref=len(ref.split()), c_err=len(ref), c_ref=len(ref), align=None)
    w = jiwer.process_words(ref, hyp)
    c = jiwer.process_characters(ref, hyp)
    return dict(w_err=w.substitutions + w.deletions + w.insertions, w_ref=len(ref.split()),
                c_err=c.substitutions + c.deletions + c.insertions, c_ref=len(ref), align=w)

def ref_word_status(al):
    """For one alignment: list of (ref word, correct?)."""
    ref = al.references[0]; status = []
    for ch in al.alignments[0]:
        span = ref[ch.ref_start_idx:ch.ref_end_idx]
        status += [(w, ch.type == "equal") for w in span]
    return status

assert normalise("ગુજરાતી, ભાષા! ૧૨") == "ગુજરાતી ભાષા 12"
assert normalise("தமிழ், ௧௨!") == "தமிழ் 12"
assert utt_errors("a b c", "a x c")["w_err"] == 1
print("normaliser + scorer ready · example:", normalise("સ્પષ્ટ ઉચ્ચારણ, ૨૦૨૪!"))

In [ ]:
# CELL: run_eval
ROWS, MODEL_INFO = [], {}
CLIPS["ref"] = CLIPS.ref_raw.map(normalise)
CLIPS = CLIPS[CLIPS.ref != ""].reset_index(drop=True)          # nothing to score against
if SMOKE_TEST:
    CLIPS_BY_ID = {id(a): s for a, s in zip(CLIPS.audio, CLIPS.ref_raw)}

def add_rows(system, sub, hyps, extra=None):
    for k, ((i, r), h) in enumerate(zip(sub.iterrows(), hyps)):
        hn = normalise(h); e = utt_errors(r.ref, hn)
        ROWS.append({"system": system, "lang": r.lang, "sid": r.sid, "gender": r.gender, "dur_s": r.dur_s,
                     "ref": r.ref, "ref_raw": r.ref_raw, "hyp_raw": h, "hyp": hn,
                     **{k2: v for k2, v in e.items() if k2 != "align"}, "alignment": e["align"],
                     "script_share": script_share(h, LANGS[r.lang][3]), **((extra or [{}] * len(hyps))[k])})

# ---- open models ----
for key, hf_id, family, licence in OPEN_MODELS:
    t_load = time.time()
    try:
        transcribe, params = (load_stand_in_asr if SMOKE_TEST else load_asr)(key, hf_id, family)
    except Exception as e:
        print(f"!! {key} failed to load: {type(e).__name__}: {str(e)[:200]}"); continue
    MODEL_INFO[key] = {"kind": "open", "id": hf_id, "licence": licence, "params_b": params, "load_s": time.time() - t_load}
    for lang in LANGS:
        sub = CLIPS[CLIPS.lang == lang].sort_values("dur_s")          # similar lengths per batch = less padding
        t = time.time(); hyps = []
        for i in range(0, len(sub), BATCH):
            hyps += transcribe(list(sub.audio.iloc[i:i + BATCH]), lang)
        wall = time.time() - t
        MODEL_INFO[key][f"xrt_{lang}"] = float(sub.dur_s.sum()) / max(wall, 1e-6)
        add_rows(key, sub, hyps)
        sel = [x for x in ROWS if x["system"] == key and x["lang"] == lang]
        print(f"{key:26s} {lang}: WER {sum(x['w_err'] for x in sel) / max(1, sum(x['w_ref'] for x in sel)):6.1%} · "
              f"{len(sel)} clips · {MODEL_INFO[key][f'xrt_{lang}']:5.1f}× real time")
    del transcribe; _free()

# ---- the API (raw transcript and edited text from the same call) ----
API_RESP, API_BILLED_MIN = {}, 0.0
if API_ON:
    t = time.time()
    API_RESP, API_BILLED_MIN = run_api(CLIPS)
    print(f"API done in {time.time() - t:.0f}s · audio sent this run {API_BILLED_MIN:.1f} min")
    ok = {i: r for i, r in API_RESP.items() if "error" not in r}
    sub = CLIPS.loc[sorted(ok)]
    meta = [{"api_lang": ok[i].get("language_code"), "degraded": ok[i].get("degraded"),
             "latency_s": ok[i].get("latency_s"), "api_model": ok[i].get("model")} for i in sub.index]
    add_rows(API_RAW, sub, [ok[i]["transcript"] for i in sub.index], meta)
    add_rows(API_EDIT, sub, [ok[i]["text"] for i in sub.index], meta)
    served = sorted({str(m["api_model"]) for m in meta})
    MODEL_INFO[API_RAW] = {"kind": "api", "id": f"{DICTATE_MODEL} (served: {', '.join(served)})",
                           "licence": f"₹{PRICE_INR_PER_MIN:.2f}/audio min (list price)", "params_b": float("nan")}
    MODEL_INFO[API_EDIT] = dict(MODEL_INFO[API_RAW])
    API_ERRORS = collections.Counter(r["error"][:60] for r in API_RESP.values() if "error" in r)
    if API_ERRORS:
        print("API errors:", dict(API_ERRORS))
    for lang in LANGS:
        s_ = sub[sub.lang == lang]; lat = [m["latency_s"] for m, i in zip(meta, sub.index) if CLIPS.lang[i] == lang]
        MODEL_INFO[API_RAW][f"xrt_{lang}"] = float(s_.dur_s.sum()) / max(sum(lat), 1e-6)   # one request at a time
        MODEL_INFO[API_EDIT][f"xrt_{lang}"] = MODEL_INFO[API_RAW][f"xrt_{lang}"]
else:
    API_ERRORS = collections.Counter()
    print("API arm skipped (no key).")

res = pd.DataFrame(ROWS)
res.drop(columns=["alignment"]).to_csv("ft12_transcripts.csv", index=False)
print("saved ft12_transcripts.csv;", len(res), "rows")

## 5 · Results: the headline number, after the sanity checks

In [ ]:
# CELL: results
def _boot(err, ref, n=N_BOOT, seed=SEED):
    r = np.random.default_rng(seed); idx = r.integers(0, len(err), (n, len(err)))
    b = err[idx].sum(1) / np.maximum(ref[idx].sum(1), 1)
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

summ = []
for (s, lang), g in res.groupby(["system", "lang"]):
    lo, hi = _boot(g.w_err.values, g.w_ref.values)
    summ.append({"system": s, "lang": lang, "clips": len(g), "WER": g.w_err.sum() / g.w_ref.sum(), "WER_lo": lo,
                 "WER_hi": hi, "CER": g.c_err.sum() / g.c_ref.sum(),
                 "WER_mean_per_clip": float((g.w_err / g.w_ref).mean()), "empty": int((g.hyp == "").sum()),
                 "script_ok": float((g.script_share >= 0.5).mean()),
                 "x_realtime": MODEL_INFO.get(s, {}).get(f"xrt_{lang}", float("nan"))})
SUMMARY = pd.DataFrame(summ)
OPEN_KEYS = [m[0] for m in OPEN_MODELS if m[0] in MODEL_INFO]
SYSTEMS = ([API_RAW] if API_RAW in MODEL_INFO else []) + OPEN_KEYS          # the headline comparison
WER = SUMMARY.pivot(index="system", columns="lang", values="WER").reindex(columns=list(LANGS))
WER_MAIN = WER.reindex(SYSTEMS)
HAS_API = API_RAW in WER.index
N_UTT = int(N_CLIPS.min())

def _best_open(lang):
    s = SUMMARY[(SUMMARY.lang == lang) & SUMMARY.system.isin(OPEN_KEYS)]
    return s.sort_values("WER").iloc[0] if len(s) else None
BEST_OPEN = {l: (_best_open(l).system if _best_open(l) is not None else "n/a") for l in LANGS}
OPEN_W = {l: (float(_best_open(l).WER) if _best_open(l) is not None else float("nan")) for l in LANGS}
API_W = {l: (float(WER.loc[API_RAW, l]) if HAS_API else float("nan")) for l in LANGS}
EDIT_W = {l: (float(WER.loc[API_EDIT, l]) if API_EDIT in WER.index else float("nan")) for l in LANGS}
WINS = sum(API_W[l] < OPEN_W[l] for l in LANGS) if HAS_API else 0

def paired_gap(lang):
    """API raw WER minus best open WER on the same clips, with a paired 95% bootstrap interval."""
    a = res[(res.system == API_RAW) & (res.lang == lang)].set_index("sid")
    o = res[(res.system == BEST_OPEN[lang]) & (res.lang == lang)].set_index("sid")
    ids = a.index.intersection(o.index)
    a, o = a.loc[ids], o.loc[ids]
    ae, oe, rr = a.w_err.values, o.w_err.values, a.w_ref.values
    r = np.random.default_rng(SEED); idx = r.integers(0, len(ids), (N_BOOT, len(ids)))
    d = (ae[idx].sum(1) - oe[idx].sum(1)) / np.maximum(rr[idx].sum(1), 1)
    return (ae.sum() - oe.sum()) / max(rr.sum(), 1), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), len(ids)
GAPS = {l: paired_gap(l) for l in LANGS} if HAS_API and OPEN_KEYS else {}
SIG_WINS = sum(g[2] < 0 for g in GAPS.values())      # API lower and the paired 95% interval excludes zero
SIG_LOSSES = sum(g[1] > 0 for g in GAPS.values())

if HAS_API:
    api_rows = res[res.system == API_RAW]
    LAT_MED = float(api_rows.latency_s.median())
    LAT_P90 = float(api_rows.latency_s.quantile(0.9))
    ed = res[res.system == API_EDIT].set_index(["lang", "sid"]); rw = api_rows.set_index(["lang", "sid"])
    CHANGED = float((ed.hyp != rw.loc[ed.index].hyp).mean())        # share of clips the editor changed (after normalising)
    _e = [utt_errors(a, b) for a, b in zip(rw.loc[ed.index].hyp, ed.hyp) if a]
    EDIT_DIST = sum(x["w_err"] for x in _e) / max(1, sum(x["w_ref"] for x in _e))   # word-level change raw → edited
else:
    LAT_MED = LAT_P90 = CHANGED = EDIT_DIST = float("nan")
INR_PER_HOUR = PRICE_INR_PER_MIN * 60
BEST_OPEN_XRT = MODEL_INFO.get(BEST_OPEN["gu_in"], {}).get("xrt_gu_in", float("nan"))

# ---------------- sanity checks (before any headline is printed) ----------------
PROBLEMS, NOTES = [], []
for lang, n in N_CLIPS.items():
    if n < N_MIN:
        PROBLEMS.append(f"only {n} usable {lang} clips (< N_MIN={N_MIN}); a download or extraction probably failed")
for lang, v in LANGS.items():
    sh = CLIPS[CLIPS.lang == lang].ref_raw.map(lambda s: script_share(s, v[3])).median()
    if sh < 0.8:
        PROBLEMS.append(f"{v[0]} references are only {sh:.0%} {v[3]} script: wrong FLEURS config or a broken TSV parse")
if N_LONG_DECODED:
    NOTES.append(f"{N_LONG_DECODED} decoded clips were longer than {MAX_AUDIO_S:g}s despite the TSV length and were dropped")
if not CLIPS.dur_s.between(0.5, 30).mean() >= 0.95:
    PROBLEMS.append("more than 5% of clips are shorter than 0.5 s: audio was decoded wrongly")
if len(OPEN_KEYS) < len(OPEN_MODELS):
    PROBLEMS.append(f"open model(s) failed to load: {[m[0] for m in OPEN_MODELS if m[0] not in OPEN_KEYS]}")
if len(SUMMARY) and not SUMMARY.WER.between(0, 5).all():
    PROBLEMS.append("a WER is negative, NaN or above 500%: the scorer is broken")
if "whisper-large-v3-turbo" in WER.index and WER.loc["whisper-large-v3-turbo", "en_us"] > EN_WER_CEILING:
    PROBLEMS.append(f"whisper-large-v3-turbo English WER is {WER.loc['whisper-large-v3-turbo', 'en_us']:.0%} "
                    f"(> {EN_WER_CEILING:.0%}) on read speech: the audio, language code or normaliser is broken")
if OPEN_W["gu_in"] < OPEN_W["en_us"]:
    PROBLEMS.append("the best open model did better in Gujarati than in English on the same sentences, which "
                    "contradicts every published FLEURS result: check the language codes and TSV columns")
if not HAS_API:
    PROBLEMS.append("the API arm did not run (no SARVAM_API_KEY, or every call failed): the comparison is the point "
                    "of this study, so there is no headline to post")
else:
    n_api = len(CLIPS)
    n_err = sum("error" in r for r in API_RESP.values()) + (n_api - len(API_RESP))
    api_rows = res[res.system == API_RAW]
    n_deg = int(api_rows.degraded.notna().sum())
    n_wrong_lang = int(sum(a is not None and a != LANGS[l][4] for a, l in zip(api_rows.api_lang, api_rows.lang)))
    if n_err > MAX_FAIL_SHARE * n_api:
        PROBLEMS.append(f"{n_err}/{n_api} API calls failed or were not sent ({dict(API_ERRORS)}): rerun (cached clips "
                        "are not billed again) before quoting")
    elif n_err:
        NOTES.append(f"{n_err} API calls failed; those clips are missing from the API rows only")
    if n_deg > MAX_FAIL_SHARE * len(api_rows):
        PROBLEMS.append(f"{n_deg} API responses came back 'degraded' (edited text = raw transcript)")
    if n_wrong_lang > MAX_FAIL_SHARE * len(api_rows):
        PROBLEMS.append(f"{n_wrong_lang} API responses report a different language than the one sent: check language_code")
    if API_W["en_us"] > EN_WER_CEILING:
        PROBLEMS.append(f"API English WER is {API_W['en_us']:.0%} (> {EN_WER_CEILING:.0%}) on read speech: the request "
                        "(audio encoding, language_code, mode) is probably wrong")
    for l in LANGS:
        if API_W[l] > API_VS_OPEN_LIMIT * OPEN_W[l] and API_W[l] > 0.10:
            PROBLEMS.append(f"API {LANGS[l][0]} WER {API_W[l]:.0%} is over {API_VS_OPEN_LIMIT:g}× the best open model's "
                            f"{OPEN_W[l]:.0%}: implausible for a system built for Indian languages; check the call first")
    if API_BILLED_MIN > MAX_API_MINUTES * 0.99:
        PROBLEMS.append("the API budget ran out before every clip was sent")
    served = MODEL_INFO[API_RAW]["id"]
    NOTES.append(f"API model {served}; API audio sent this run {API_BILLED_MIN:.1f} min "
                 f"(≈ ₹{API_BILLED_MIN * PRICE_INR_PER_MIN:.0f} at list price)")
if not PARALLEL:
    NOTES.append("languages were sampled separately (not the same sentences)")
if not HAS_GPU and not SMOKE_TEST:
    NOTES.append("CPU run: open-model speeds are not comparable with a T4")
if SMOKE_TEST:
    PROBLEMS.append("SMOKE_TEST is on: synthetic clips, stand-in models and a stand-in API")
for n_ in NOTES:
    print("Note:", n_)
if PROBLEMS:
    print("!" * 100)
    for p in PROBLEMS: print("!! SANITY CHECK FAILED:", p)
    print("!! Do NOT post these numbers until every check above passes.")
    print("!" * 100)
else:
    print("All sanity checks passed.")

def _pct(x): return f"{100 * x:.0f}%" if np.isfinite(x) else "n/a"
if HAS_API:
    HEADLINE = (f"Sarvam Dictate's raw transcript got {_pct(API_W['gu_in'])} of Gujarati words wrong on {N_UTT} public "
                f"test clips; the best free open model ({BEST_OPEN['gu_in']}) got {_pct(OPEN_W['gu_in'])}. Lower error in "
                f"{WINS} of {len(LANGS)} languages, clearly lower in {SIG_WINS}.")
    # "clearly lower" = API lower and the paired 95% bootstrap interval of the gap is entirely below zero
else:
    HEADLINE = (f"(API arm skipped) The best free open model ({BEST_OPEN['gu_in']}) got {_pct(OPEN_W['gu_in'])} "
                f"of Gujarati words wrong on {N_UTT} FLEURS test clips.")
print()
print("HEADLINE (estimate, FLEURS test clips):", HEADLINE)
for l, v in LANGS.items():
    g = GAPS.get(l)
    print(f"  {v[0]:9s} API raw {_pct(API_W[l]):>4s} · API edited {_pct(EDIT_W[l]):>4s} · best open {_pct(OPEN_W[l]):>4s} "
          f"({BEST_OPEN[l]})" + (f" · gap {100 * g[0]:+.1f} pts (paired 95% CI {100 * g[1]:+.1f} to {100 * g[2]:+.1f}, {g[3]} clips)" if g else ""))
if HAS_API:
    print(f"  API latency per clip: median {LAT_MED:.2f}s, p90 {LAT_P90:.2f}s · list price ₹{INR_PER_HOUR:.0f} per audio hour")
    print(f"  Editor changed {CHANGED:.0%} of clips; word-level change raw → edited {EDIT_DIST:.0%}")
print(f"  Best open model on Gujarati runs at {BEST_OPEN_XRT:.0f}× real time on {HARDWARE} (free tier)")
print("  Robustness (mean of per-clip WER instead of pooled):")
print(SUMMARY.pivot(index="system", columns="lang", values="WER_mean_per_clip").reindex(columns=list(LANGS)).round(3).to_string())
print("  Robustness (CER):")
print(SUMMARY.pivot(index="system", columns="lang", values="CER").reindex(columns=list(LANGS)).round(3).to_string())
print(f"  Assumptions: N_PER_LANG={N_PER_LANG}, SEED={SEED}, MAX_AUDIO_S={MAX_AUDIO_S}, greedy decoding, "
      f"DICTATE_MODEL={DICTATE_MODEL}, mode={DICTATE_MODE}, preset={DICTATE_PRESET}, PRICE_INR_PER_MIN={PRICE_INR_PER_MIN}, "
      f"EN_WER_CEILING={EN_WER_CEILING}, API_VS_OPEN_LIMIT={API_VS_OPEN_LIMIT}, N_BOOT={N_BOOT}")
print()
print(SUMMARY.round(3).to_string(index=False))
SUMMARY.to_csv("ft12_summary.csv", index=False)

In [ ]:
# CELL: results_chart
SYS_COLORS = {API_RAW: B.MOSS, "whisper-large-v3-turbo": B.AMBER, "mms-1b-all": B.INK, API_EDIT: B.STONE}
SHORT = {API_RAW: "Sarvam Dictate (raw)", API_EDIT: "Sarvam Dictate (edited)",
         "whisper-large-v3-turbo": "whisper-large-v3-turbo", "mms-1b-all": "mms-1b-all"}
def draw_results(ax, big=False, grow=1.0, systems=None):
    systems = systems or SYSTEMS; langs = list(LANGS)
    x = np.arange(len(langs)); w = 0.8 / max(1, len(systems))
    vmax = np.nanmax(WER.reindex(systems).values) if len(systems) else 1
    ymax = float(vmax) * 1.25 if np.isfinite(vmax) and vmax > 0 else 1
    for j, s in enumerate(systems):
        vals = WER.loc[s, langs].values
        bars = ax.bar(x + (j - (len(systems) - 1) / 2) * w, np.nan_to_num(vals) * 100 * grow, w * 0.92,
                      color=SYS_COLORS.get(s, B.STONE), label=SHORT.get(s, s))
        for b, v in zip(bars, vals):
            if np.isfinite(v) and grow > 0.95:
                ax.text(b.get_x() + b.get_width() / 2, v * 100 + ymax * 1.5, f"{100 * v:.0f}", ha="center",
                        va="bottom", fontsize=14 if big else 9, color=B.INK, fontproperties=B.FP.get("sans_bold"))
    ax.set_xticks(x, [LANGS[l][0] for l in langs], fontsize=16 if big else 10)
    ax.set_ylim(0, ymax * 100); ax.set_ylabel("word error rate, % (lower is better)", fontsize=13 if big else 10)
    ax.tick_params(axis="y", labelsize=12 if big else 9)
    ax.legend(loc="upper left", fontsize=13 if big else 9, ncol=1, frameon=False)

fig, ax = plt.subplots(figsize=(9.5, 4.8))
draw_results(ax, systems=[s for s in SYSTEMS + [API_EDIT] if s in WER.index])
B.chart_title(ax, "Same clips, every system", f"FLEURS test · {N_UTT} clips per language · edited text shown for reference")
B.source_note(fig, "23N Field Test #12 · " + DATA_CREDIT + " · Independent study on public data.")
plt.tight_layout(); fig.savefig("ft12_results.png", dpi=150); plt.show()

## 6 · Where the errors are (Gujarati), and what the editing step changes

Every reference word is marked right or wrong from the alignment and grouped into **numbers**, **Latin-script
words** (names and English terms) and **other words**. For the API, the notebook also shows clips where the edited
`text` differs from the raw `transcript`, so you can judge whether the editor's changes are fixes or rewrites.

In [ ]:
# CELL: failure_analysis
def word_kind(w):
    return "numbers" if re.search(r"\d", w) else ("Latin-script words" if re.search(r"[a-z]", w) else "other words")

fail = []
for s, g in res[res.lang == "gu_in"].groupby("system"):
    cnt = collections.Counter(); bad = collections.Counter()
    for _, r in g.iterrows():
        st = [(w, False) for w in r["ref"].split()] if r["alignment"] is None else ref_word_status(r["alignment"])
        for w, ok in st:
            k = word_kind(w); cnt[k] += 1; bad[k] += (not ok)
    for k in ("numbers", "Latin-script words", "other words"):
        fail.append({"system": s, "kind": k, "words": cnt[k], "error_rate": bad[k] / cnt[k] if cnt[k] else float("nan")})
FAIL = pd.DataFrame(fail)
print("Gujarati: share of reference words wrong, by kind of word")
print(FAIL.pivot(index="system", columns="kind", values="error_rate").round(3).to_string())
print(FAIL.groupby("kind").words.first().to_string())

if HAS_API:
    ed = res[res.system == API_EDIT].set_index(["lang", "sid"]); rw = res[res.system == API_RAW].set_index(["lang", "sid"])
    diff = [(k, rw.loc[k].hyp_raw, ed.loc[k].hyp_raw, ed.loc[k].ref_raw) for k in ed.index if rw.loc[k].hyp != ed.loc[k].hyp]
    print(f"\nEditor changed {len(diff)} of {len(ed)} clips. Up to 6 examples (raw → edited | reference):")
    for k, a, b, c in diff[:6]:
        print(f"  [{k[0]} {k[1]}]\n    raw:    {a}\n    edited: {b}\n    ref:    {c}")

fig, ax = plt.subplots(figsize=(9, 4.2))
piv = FAIL.pivot(index="system", columns="kind", values="error_rate").reindex(SYSTEMS)[["numbers", "Latin-script words", "other words"]]
x = np.arange(len(piv)); w = 0.26
for j, k in enumerate(piv.columns):
    ax.bar(x + (j - 1) * w, np.nan_to_num(piv[k].values) * 100, w * 0.92, color=B.SERIES_COLORS[j], label=k)
ax.set_xticks(x, [SHORT.get(s, s) for s in piv.index], fontsize=9)
ax.set_ylabel("reference words wrong, %"); ax.legend(fontsize=9)
B.chart_title(ax, "Which Gujarati words go wrong", "share of reference words missed or swapped")
B.source_note(fig, "23N Field Test #12 · " + DATA_CREDIT + " · Independent study on public data.")
plt.tight_layout(); fig.savefig("ft12_failures.png", dpi=150); plt.show()

## 7 · Interactive demo
Record or upload a short clip (up to 30 s) and compare what the API and the best open model write. Each API call
uses your credit (₹0.50 per minute at list price). The cell doesn't block "Run all"; open the `gradio.live` link it
prints. Audio goes to this Colab session and, for the API column, to Sarvam's API.

In [ ]:
# CELL: gradio
try:
    if SMOKE_TEST:
        raise RuntimeError("skipped in smoke test")
    import gradio as gr
    _m = [m for m in OPEN_MODELS if m[0] == BEST_OPEN["gu_in"]][0]
    DEMO_ASR, _ = load_asr(_m[0], _m[1], _m[2])
    def gr_compare(path, lang_label):
        if not path:
            return "Record or upload a clip first."
        a, sr = sf.read(path, dtype="float32")
        a = to_16k(a.mean(axis=1) if a.ndim > 1 else a, sr)[: int(MAX_AUDIO_S * 16000)]
        lang = {v[0]: k for k, v in LANGS.items()}[lang_label]
        t = time.time(); open_txt = DEMO_ASR([a], lang)[0]; t_open = time.time() - t
        out = f"**{_m[0]}** ({t_open:.1f}s, free GPU)\n\n{open_txt}\n\n"
        if SARVAM_KEY:
            r = dictate_call(a, lang)
            out += (f"**Sarvam Dictate raw** ({r.get('latency_s', 0):.1f}s)\n\n{r.get('transcript', r.get('error'))}\n\n"
                    f"**Sarvam Dictate edited**\n\n{r.get('text', '')}\n\n")
        return out + "<sub>Independent study on public data by 23N AI Labs</sub>"
    with gr.Blocks(title="23N Field Test #12 · Dictation in Indian languages") as demo:
        gr.Markdown("## Same clip, two systems\nRecord a sentence and compare.")
        au = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Clip (up to 30 s)")
        lg = gr.Dropdown([v[0] for v in LANGS.values()], value="Gujarati", label="Language")
        out = gr.Markdown()
        gr.Button("Transcribe").click(gr_compare, [au, lg], out)
    demo.launch(share=True, prevent_thread_lock=True)
except Exception as e:
    print("Gradio demo not started:", e)

## 8 · Video (1080×1080, 30 fps, 58 s timeline saved at 1.5×, H.264) and cover (1080×1350)
Every number and transcript in the video comes from the run above. The example is the first Gujarati clip (in
sentence-ID order) of 6–16 words where the API's raw WER is between 10% and 60%, so it shows a typical clip rather
than the best or worst case.

In [ ]:
# CELL: video_lib
VIDEO_PATH, COVER_PATH = f"ft{FT_NUM}_video.mp4", f"ft{FT_NUM}_cover.png"
FPS = 30
def _ease(p): p = min(max(p, 0.0), 1.0); return 1 - (1 - p) ** 3
def _prog(t, a, b): return min(max((t - a) / (b - a), 0.0), 1.0)
LEAD = API_RAW if HAS_API else BEST_OPEN["gu_in"]

def pick_example():
    g = res[(res.lang == "gu_in") & (res.system == LEAD)].copy()
    g["wer"] = g.w_err / g.w_ref
    c = g[g.w_ref.between(6, 16) & g.wer.between(0.10, 0.6)].sort_values("sid")
    row = c.iloc[0] if len(c) else g.iloc[(g.wer - g.wer.median()).abs().argsort().iloc[0]]
    hyps = res[(res.lang == "gu_in") & (res.sid == row.sid)].set_index("system")
    order = [s for s in [API_RAW, API_EDIT] + OPEN_KEYS if s in hyps.index]
    return row, [(s, hyps.loc[s]) for s in order]

def _short(s, n=110):
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s if len(s) <= n else s[:n].rsplit(" ", 1)[0] + " …"

def draw_evidence(fig, t, ctx):
    """t runs 0..35 inside the evidence section (video seconds 5..40)."""
    B.header(fig, FT_NUM)
    if t < 7:                                                         # the test set
        fig.text(0.055, 0.85, "The test", fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
        fig.text(0.055, 0.785, ("SAME SENTENCES, READ ALOUD IN 4 LANGUAGES" if PARALLEL else "FLEURS TEST CLIPS, 4 LANGUAGES"),
                 fontproperties=B.FP.get("mono"), fontsize=15, color=B.MUTED, va="top")
        for i, lang in enumerate(LANGS):
            a = _prog(t, 0.4 + i * 0.7, 1.1 + i * 0.7); y = 0.68 - i * 0.125
            fig.text(0.055, y, LANGS[lang][0], fontproperties=B.FP.get("display"), fontsize=42, color=B.INK, alpha=a, va="top")
            fig.text(0.52, y, f"{int(DATA_TABLE.loc[lang, 'clips'])} clips", fontproperties=B.FP.get("display"), fontsize=36,
                     color=B.INK, alpha=a, va="top")
            fig.text(0.52, y - 0.05, f"{DATA_TABLE.loc[lang, 'minutes']:.0f} MIN OF AUDIO", fontproperties=B.FP.get("mono"),
                     fontsize=13, color=B.MUTED, alpha=a, va="top")
        B.caption(fig, ("A dictation API and " if HAS_API else "") + f"{len(OPEN_KEYS)} free open models · every clip ≤ 30 s")
    elif t < 20:                                                      # one clip, every system
        row, hyps = ctx["example"]
        fig.text(0.055, 0.85, "WHAT WAS SAID", fontproperties=B.FP.get("mono_bold"), fontsize=14, color=B.MUTED, va="top")
        put_text_img(fig, _short(row.ref_raw, 120), 0.055, 0.82, 34, B.INK, bold=True, max_width=970, max_lines=2)
        for i, (s, h) in enumerate(hyps):
            t0 = 8.5 + i * 2.2
            if t < t0:
                continue
            a = _prog(t, t0, t0 + 0.6); y = 0.655 - i * 0.12
            wer = h.w_err / h.w_ref
            col = SYS_COLORS.get(s, B.STONE)
            fig.text(0.055, y, f"{SHORT.get(s, s).upper()} · WER {100 * wer:.0f}%", fontproperties=B.FP.get("mono_bold"),
                     fontsize=13, color=B.MOSS if s == LEAD else B.MUTED, alpha=a, va="top")
            put_text_img(fig, _short(h.hyp_raw, 90) or "(no output)", 0.055, y - 0.025, 25,
                         B.MOSS if s == LEAD else B.BODY, max_width=970, alpha=a, max_lines=2)
        B.caption(fig, "One Gujarati test clip and what each system wrote")
    elif t < 29:                                                      # the scoreboard
        fig.text(0.055, 0.85, "Word error rate", fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
        fig.text(0.055, 0.785, f"% OF WORDS WRONG · LOWER IS BETTER · {N_UTT} CLIPS PER LANGUAGE",
                 fontproperties=B.FP.get("mono"), fontsize=14, color=B.MUTED, va="top")
        ax = fig.add_axes([0.10, 0.20, 0.85, 0.52]); draw_results(ax, big=True, grow=_ease(_prog(t, 20.3, 23)))
        B.caption(fig, (f"The API had the lower error in {WINS} of {len(LANGS)} languages ({SIG_WINS} clearly)" if HAS_API
                        else "Open models only: the API arm was skipped"))
    else:                                                             # cost and speed
        fig.text(0.055, 0.85, "Cost and speed", fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
        fig.text(0.055, 0.785, "LIST PRICE AND THIS RUN'S TIMINGS", fontproperties=B.FP.get("mono"), fontsize=14,
                 color=B.MUTED, va="top")
        a1, a2 = _prog(t, 29.3, 30.0), _prog(t, 30.3, 31.0)
        tiles = [(0.055, a1, "API", f"₹{INR_PER_HOUR:.0f}" if HAS_API else "n/a", "PER HOUR OF AUDIO",
                  f"median {LAT_MED:.1f}s per clip" if HAS_API else "skipped"),
                 (0.52, a2, "OPEN MODEL", f"{BEST_OPEN_XRT:.0f}×" if np.isfinite(BEST_OPEN_XRT) else "n/a",
                  "FASTER THAN REAL TIME", f"{BEST_OPEN['gu_in']} · free {HARDWARE}")]
        for x, a, lab, big, unit, sub_ in tiles:
            fig.patches.append(matplotlib.patches.FancyBboxPatch((x, 0.30), 0.425, 0.42, boxstyle="round,pad=0.005,rounding_size=0.015",
                               transform=fig.transFigure, fc=B.PAPER_DEEP, ec="none", alpha=a))
            fig.text(x + 0.03, 0.68, lab, fontproperties=B.FP.get("mono_bold"), fontsize=15, color=B.MUTED, alpha=a, va="top")
            fig.text(x + 0.03, 0.60, big, fontproperties=B.FP.get("display"), fontsize=80, color=B.MOSS if lab == "API" else B.INK,
                     alpha=a, va="top")
            fig.text(x + 0.03, 0.44, unit, fontproperties=B.FP.get("mono"), fontsize=13, color=B.MUTED, alpha=a, va="top")
            fig.text(x + 0.03, 0.39, textwrap.fill(sub_, 30), fontproperties=B.FP.get("sans"), fontsize=15, color=B.INK, alpha=a, va="top")
        B.caption(fig, "The API needs no GPU; the open model needs one, but no per-minute fee")

def render_video(ctx, path=VIDEO_PATH):
    v = B.Video(path, fps=FPS)
    B.hook_card(v.fig, FT_NUM, "Dictation in Indian languages: paid API or free model?",
                f"FLEURS test · Gujarati, Hindi, Tamil, English · {N_UTT} clips each"); v.hold(5)
    n = int(35 * FPS); key0, buf = None, None
    for f in range(n):
        t = f / FPS
        key = (("d", min(round(t * 10) / 10, 3.5)) if t < 7 else ("e", round(t * 10) / 10) if t < 20 else
               ("s", min(round(t * 15) / 15, 23.1)) if t < 29 else ("c", min(round(t * 15) / 15, 31.1)))
        if key != key0:
            v.clear(); draw_evidence(v.fig, t, ctx); buf = v.grab(); key0 = key
        v.proc.stdin.write(buf); v.frames += 1
    if HAS_API:
        stmt = f"of Gujarati words wrong (API, raw). Best free model: {_pct(OPEN_W['gu_in'])}"
        num = _pct(API_W["gu_in"])
    else:
        stmt, num = "of Gujarati words wrong for the best free open model", _pct(OPEN_W["gu_in"])
    v.clear(); B.headline_card(v.fig, FT_NUM, num, stmt,
                               f"FLEURS test · {N_UTT} clips per language · word error rate · raw transcript"); v.hold(12)
    v.clear(); B.end_card(v.fig, FT_NUM, DATA_CREDIT); v.hold(6)
    v.close()
    return path

def render_cover(path=COVER_PATH):
    num = _pct(API_W["gu_in"]) if HAS_API else _pct(OPEN_W["gu_in"])
    cap = (f"of Gujarati words wrong for the API's raw transcript. Best open model: {_pct(OPEN_W['gu_in'])}" if HAS_API
           else "of Gujarati words wrong for the best open model")
    B.cover(path, FT_NUM, "Dictation in Indian languages: paid API or free model?",
            "same public clips · Gujarati, Hindi, Tamil, English", num, cap,
            image=lambda ax: draw_results(ax.inset_axes([0.09, 0.1, 0.91, 0.9]), big=True), data_credit=DATA_CREDIT)
    return path

In [ ]:
# CELL: video_render
VCTX = {"example": pick_example()}
render_video(VCTX)
render_cover()
_probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                         "stream=width,height,r_frame_rate,codec_name:format=duration", "-of", "json", VIDEO_PATH],
                        capture_output=True, text=True)
print(_probe.stdout)
try:
    from google.colab import files
    files.download(VIDEO_PATH); files.download(COVER_PATH)
except Exception:
    print("Not on Colab: video and cover saved in the working directory.")

## 9 · Post text (ready to paste once the sanity checks pass)

In [ ]:
# CELL: post_text
POST_LINKEDIN_TEMPLATE = """Dictation in Indian languages: does a paid API beat free open models?

23N Field Test #12: {headline}

On 23 Nov a dictation API for English and 22 Indian languages launched at ₹0.50 per audio minute. Its maker reports fewer edits than rival systems on its own, non-public test set. We ran it on a public one.

How we tested it:
• {n_utt} FLEURS test clips per language (Gujarati, Hindi, Tamil, English), the same sentences in each
• Sarvam Dictate's raw transcript vs {open_list} on a free Colab GPU
• Word error rate, API vs best open: Gujarati {api_gu} vs {open_gu}, Hindi {api_hi} vs {open_hi}, Tamil {api_ta} vs {open_ta}, English {api_en} vs {open_en}
• Cost: ₹{inr_hr} per audio hour; median {lat}s per clip

Limits: read speech, not real dictation; the API's edited text is scored separately. Not affiliated with Sarvam AI.

Notebook in the comments.

{signature}
"""
POST_X_TEMPLATE = """Sarvam Dictate vs free open speech models, same public FLEURS clips. Word error rate, Gujarati: {api_gu} (API, raw) vs {open_gu} ({best_open_gu}). Hindi {api_hi} vs {open_hi}. Tamil {api_ta} vs {open_ta}. 23N Field Test #12

---
Thread (optional):
1/ {n_utt} clips per language, language stated to every system, one script-safe normaliser, paired bootstrap on the gap.
2/ The API's edited text differed from its raw transcript on {changed} of clips. Gujarati WER: {edit_gu} edited vs {api_gu} raw.
3/ Limits: read speech, not real dictation. ₹{inr_hr}/audio hour list price. Independent study on public data.
"""
def _load_template(name, fallback):
    try:
        return open(name, encoding="utf-8").read()
    except Exception:
        return fallback
P = {f"api_{l[:2]}": _pct(API_W[l]) for l in LANGS} | {f"open_{l[:2]}": _pct(OPEN_W[l]) for l in LANGS}
POST_VALUES = dict(headline=HEADLINE, n_utt=N_UTT, open_list=" and ".join(OPEN_KEYS), best_open_gu=BEST_OPEN["gu_in"],
                   inr_hr=f"{INR_PER_HOUR:.0f}", lat=f"{LAT_MED:.1f}", changed=_pct(CHANGED), edit_gu=_pct(EDIT_W["gu_in"]),
                   signature=B.post_signature(FT_NUM) + " #SpeechRecognition #IndianLanguages #OpenSourceAI", **P)
post_li = _load_template("post_linkedin.md", POST_LINKEDIN_TEMPLATE).format(**POST_VALUES)
post_x = _load_template("post_x.md", POST_X_TEMPLATE).format(**POST_VALUES)
x_main = post_x.split("---")[0].strip()
if PROBLEMS:
    print("!! SANITY CHECKS FAILED (see Results). Do NOT post this text.\n")
print("=" * 30, f"LinkedIn ({len(post_li)} chars)", "=" * 30); print(post_li)
print("=" * 30, f"X (main post {len(x_main)} chars)", "=" * 30); print(post_x)
if len(post_li) > 1300: print(f"WARNING: LinkedIn post is {len(post_li)} chars (> 1,300): trim before posting")
if len(x_main) > 280: print(f"WARNING: X post is {len(x_main)} chars (> 280): trim before posting")

## 10 · Save to Drive
Copies the video, cover, charts, CSVs, both post texts, the API response cache (no key in it) and a `RUN_STATUS.txt`
with the headline and every sanity-check result into `23N Field Tests/FT12/<date_time>` in your Drive. A run whose
sanity checks failed gets `_CHECKS-FAILED` in the folder name, and a smoke test gets `_SMOKE`, so they can't be
mistaken for a postable run.

In [ ]:
# CELL: save_to_drive
import shutil
open(f"ft{FT_NUM}_post_linkedin.txt", "w", encoding="utf-8").write(post_li)
open(f"ft{FT_NUM}_post_x.txt", "w", encoding="utf-8").write(post_x)
STAMP = time.strftime("%Y-%m-%d_%H%M") + ("_SMOKE" if SMOKE_TEST else "_CHECKS-FAILED" if PROBLEMS else "")
status = [f"23N Field Test #{FT_NUM} run {STAMP}", f"Hardware: {HARDWARE}",
          f"API: {MODEL_INFO[API_RAW]['id'] if HAS_API else 'skipped'} · audio sent this run {API_BILLED_MIN:.1f} min",
          "", "HEADLINE: " + HEADLINE, "",
          "SANITY CHECKS: " + ("ALL PASSED" if not PROBLEMS else "FAILED, do NOT post")]
status += [f"  !! {p}" for p in PROBLEMS] + [f"  note: {n}" for n in NOTES]
status += ["", SUMMARY.round(3).to_string(index=False)]
open("RUN_STATUS.txt", "w", encoding="utf-8").write("\n".join(status) + "\n")
OUTPUTS = [VIDEO_PATH, COVER_PATH, f"ft{FT_NUM}_results.png", f"ft{FT_NUM}_failures.png", f"ft{FT_NUM}_summary.csv",
           f"ft{FT_NUM}_transcripts.csv", f"ft{FT_NUM}_post_linkedin.txt", f"ft{FT_NUM}_post_x.txt", CACHE_PATH,
           "RUN_STATUS.txt"]
if DRIVE_OK:
    dest = os.path.join(SAVE_DIR_OVERRIDE or DRIVE_ROOT, f"FT{FT_NUM}", STAMP)
    os.makedirs(dest, exist_ok=True)
    saved = []
    for f in OUTPUTS:
        if os.path.exists(f):
            shutil.copy2(f, dest); saved.append(f)
    print(f"Saved {len(saved)} files to {dest}")
    print("  " + ", ".join(saved))
    missing = [f for f in OUTPUTS if f not in saved]
    if missing:
        print("  (not produced in this run:", ", ".join(missing) + ")")
else:
    print("Drive not mounted: outputs are in the Colab file panel only and disappear when the session ends.")

## 11 · Limitations (read before quoting the number)

- **Read speech, not dictation.** FLEURS is adult volunteers reading Wikipedia-style sentences into a good
  microphone. Real dictation is spontaneous, noisier and full of names, code-mixing and self-corrections, which is
  what the API's editing step is built for. This test measures the recognition step on clean read speech; it says
  nothing about the vendor's own (non-public) benchmark, which used a different test set and metric.
- **Raw transcript, not the product's final output.** The headline scores the raw `transcript`. The edited `text`
  is what users of the API would normally see; WER against a verbatim reference penalises fair edits (punctuation is
  removed by our normaliser, but number formats, dropped fillers and rewrites are not), so its WER is shown only for
  reference.
- **US-English speakers, `en-IN` setting.** FLEURS English is `en_us`; the API is called with `en-IN`.
- **A small sample.** Up to 100 clips per language. Use the paired bootstrap interval: a gap whose interval crosses
  zero is not a difference.
- **Snapshots and prices change.** The served model snapshot and the list price are printed with the result; both
  can change after this run.
- **Licences and terms.** FLEURS is CC BY 4.0. `facebook/mms-1b-all` is **CC-BY-NC 4.0 (non-commercial)**; Whisper
  is MIT. The API is used under Sarvam's terms of service with the user's own key; check them before publishing
  comparisons.